# D25 / D50 Accuracy Analysis

This notebook reads every `d25` and `d50` eval shell script from the moved archive in `tests/data/` plus any matching workspace scripts, then combines them with the historical success-rate summary in `jobs/eval/hi/PLANNING_HPARAM_RESULTS.md`.

The main output is a sortable table of planning parameters and accuracy.

In [ ]:
from pathlib import Path

try:
    import pandas as pd
except ModuleNotFoundError:
    pd = None

from tests.d25_d50_accuracy_analysis import build_analysis_frames, build_analysis_rows

repo_root = Path.cwd()

if pd is not None:
    pd.set_option("display.max_columns", 50)
    pd.set_option("display.max_colwidth", 120)
    scripts_df, results_df, matched_df = build_analysis_frames(repo_root)
else:
    scripts_df, results_df, matched_df = build_analysis_rows(repo_root)

print(f"Loaded {len(scripts_df)} script configurations")
print(f"Loaded {len(results_df)} historical result rows")

In [ ]:
if pd is None:
    sorted(results_df, key=lambda row: (row['goal_offset_steps'], -(row['success_rate_pct'] or -1)))[:10]
else:
summary_cols = [
    "goal_offset_steps",
    "mode",
    "success_rate_pct",
    "checkpoint",
    "high_horizon",
    "high_replan_interval",
    "high_num_samples",
    "high_n_steps",
    "high_topk",
    "low_horizon",
    "low_action_block",
    "low_num_samples",
    "low_n_steps",
    "low_topk",
    "eval_budget",
    "slurm_out_file",
]

    results_df[summary_cols].sort_values(
    ["goal_offset_steps", "success_rate_pct"], ascending=[True, False]
    ).reset_index(drop=True)

In [ ]:
if pd is None:
    {
        d: max(
            [row for row in results_df if row["goal_offset_steps"] == d and row["success_rate_pct"] is not None],
            key=lambda row: row["success_rate_pct"],
        )
        for d in sorted({row['goal_offset_steps'] for row in results_df})
    }
else:
    best_by_d = (
    results_df.dropna(subset=["success_rate_pct"])
    .sort_values(["goal_offset_steps", "success_rate_pct"], ascending=[True, False])
    .groupby("goal_offset_steps", as_index=False)
    .first()
    )

    best_by_d[[
    "goal_offset_steps",
    "success_rate_pct",
    "mode",
    "high_horizon",
    "high_replan_interval",
    "low_horizon",
    "low_action_block",
    "low_topk",
    "eval_budget",
    "slurm_out_file",
    ]]

In [ ]:
if pd is None:
    matched_df[:15]
else:
matched_cols = [
    "goal_offset_steps",
    "success_rate_pct",
    "mode",
    "script_path",
    "script_name",
    "source_kind",
    "slurm_out_file",
]

    matched_df[matched_cols].sort_values(
    ["goal_offset_steps", "success_rate_pct", "script_path"],
    ascending=[True, False, True],
    ).reset_index(drop=True)

In [ ]:
if pd is None:
    scripts_df[:15]
else:
script_cols = [
    "goal_offset_steps",
    "script_path",
    "source_kind",
    "parent_script",
    "run_name",
    "checkpoint_epoch",
    "high_horizon",
    "high_replan_interval",
    "high_num_samples",
    "high_n_steps",
    "high_topk",
    "low_horizon",
    "low_action_block",
    "low_num_samples",
    "low_n_steps",
    "low_topk",
    "eval_budget",
]

    scripts_df[script_cols].reset_index(drop=True)